# Session 5 — Vector data with GeoPandas
*Séance 5 — Données vectorielles avec GeoPandas*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/05_geopandas_vectoriel.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

**By the end of this session you will be able to:**

1. Read, inspect and map vector data (points, lines, polygons) with GeoPandas.
2. Explain coordinate reference systems (CRS) and reproject data before measuring areas and distances.
3. Combine layers with spatial predicates, spatial joins, buffers and dissolve.
4. Prepare an administrative boundary (South African ward) for Earth Engine.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

countries = gpd.read_file(DATA / "naturalearth" / "ne_110m_admin_0_countries.zip")
cities = gpd.read_file(DATA / "naturalearth" / "ne_110m_populated_places.zip")
rivers = gpd.read_file(DATA / "naturalearth" / "ne_50m_rivers_lake_centerlines.zip")
countries.head()

A **GeoDataFrame** is a pandas DataFrame with a `geometry` column and a **CRS**.

In [ ]:
print(countries.crs)
ax = countries.plot(column="continent", figsize=(10, 5), legend=True, legend_kwds={"loc": "lower left", "fontsize": 7})
rivers.plot(ax=ax, color="steelblue", linewidth=0.5)
cities.plot(ax=ax, color="black", markersize=4)
ax.set_axis_off(); plt.show()

### Coordinate reference systems
`EPSG:4326` stores **degrees** of latitude/longitude: fine for display, wrong for areas. To measure, reproject to an
**equal-area** CRS (e.g. Mollweide `ESRI:54009`) or a local **UTM** zone in metres (South Africa's Western Cape: `EPSG:32734`).

In [ ]:
area_deg = countries.set_index("name").geometry.area            # meaningless "square degrees"
area_km2 = countries.to_crs("ESRI:54009").set_index("name").geometry.area / 1e6
print(area_km2.loc[["Canada", "South Africa", "Cameroon"]].round(-3))

### Spatial relationships and joins

In [ ]:
south_africa = countries.loc[countries["name"] == "South Africa", "geometry"].iloc[0]
print(cities[cities.within(south_africa)])

joined = gpd.sjoin(cities, countries[["name", "continent", "geometry"]], predicate="within", how="left",
                   lsuffix="city", rsuffix="country")
joined.groupby("continent").size().sort_values(ascending=False)

### Buffers and dissolve

In [ ]:
africa = countries[countries["continent"] == "Africa"]
continents = countries.dissolve(by="continent", aggfunc={"pop_est": "sum"})
continents[["pop_est"]].sort_values("pop_est", ascending=False)

### Our study areas: South African wards
The course repository contains two ward boundaries from the South African administrative units (ADM4).

In [ ]:
ward = gpd.read_file(DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA1011001.shp")
print(ward.crs, ward[["ADM4_PCODE", "ADM3_EN", "ADM2_EN", "ADM1_EN"]].iloc[0].to_dict())
ward.plot(edgecolor="black", facecolor="none"); plt.title("Ward ZA1011001"); plt.show()

## Part B — Lab

**Task 1 — Count.** Store the number of countries per continent in a Series `n_by_continent` (index = continent).

In [ ]:
# YOUR CODE HERE
n_by_continent = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check_equal(int(n_by_continent["Africa"]), 51)

**Task 2 — Areas done right.** Reproject the ward to UTM zone 34S (`EPSG:32734`) into `ward_utm`, then compute its
area in km² (`ward_km2`, a float).

In [ ]:
# YOUR CODE HERE
ward_utm = None
ward_km2 = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(ward_utm.crs.to_epsg() == 32734)
check(195 < ward_km2 < 210, f"Unexpected area: {ward_km2:.1f} km²")

**Task 3 — Rivers of Africa.** Keep only the river segments that **intersect** African countries (`african_rivers`), and compute
their total length in km (`river_km`) in the Mollweide CRS (`ESRI:54009`).

In [ ]:
# YOUR CODE HERE
african_rivers = None
river_km = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(len(african_rivers) > 10)
check(river_km > 10_000, "Africa has tens of thousands of km of major rivers")

**Task 4 — Prepare for Earth Engine.** Build a 1 km buffer around the ward (in UTM), reproject it back to `EPSG:4326`
into `aoi`, and save it as `aoi_ward.geojson`.

In [ ]:
# YOUR CODE HERE
aoi = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
import pathlib
check(pathlib.Path("aoi_ward.geojson").exists())
check(aoi.crs.to_epsg() == 4326)

**❓ Question —** Why do we buffer the study area in UTM rather than in EPSG:4326? What would `buffer(1000)` mean in degrees?

*Your answer here.*

## Summary

- GeoDataFrames combine attributes, geometry and a CRS.
- Reproject to a metric or equal-area CRS before measuring.
- Spatial joins and predicates answer 'what is where' questions between layers.

**To go further:**

- GeoPandas user guide: https://geopandas.org/en/stable/docs/user_guide.html

---
*Inspired by the geopandas-tutorial of Joris Van den Bossche (BSD-3-Clause) and Natural Earth data (public domain). South African boundaries: OCHA/HDX administrative units.*